# CMSC 173 &middot; Machine Learning &mdash; Week 5 Lab
## Exploratory Data Analysis: Know Your Data Before You Model

Before any model, you *look* at your data: its shape, its gaps, its outliers, and how its
columns relate. This lab is a guided tour of the core EDA moves in **pandas** &mdash; describe,
plot distributions, find and fill missing values, catch outliers, and read a correlation
heatmap &mdash; on a small student dataset you build yourself.

**How this lab works.** Each part = a short **plain-English explainer**, a **code cell**
you run, a **line-by-line walkthrough** of what it did, and an **Answer here** box. The
code does the maths; we *graph* the results so you can see what is going on.

**pandas + NumPy + Matplotlib.** **Not graded.** About 55 minutes.

---
## Part 0 &middot; Setup + build the dataset

We create a small, messy-on-purpose dataset of 200 students so every EDA technique has something
to find. Run it; you don't need to understand every line of the data-making.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
rng = np.random.default_rng(173)

n = 200
study = rng.uniform(0, 5, n)
sleep = rng.uniform(4, 9, n)
allowance = rng.normal(120, 40, n).clip(40, None)          # pesos/day
score = (45 + 7*study + 2*sleep + rng.normal(0, 6, n)).clip(0, 100)
barangay = rng.choice(['Lahug', 'Mabolo', 'Guadalupe', 'Talamban'], n)
df = pd.DataFrame({'study_hours': study, 'sleep_hours': sleep,
                   'allowance': allowance, 'quiz_score': score, 'barangay': barangay})
df.loc[rng.choice(n, 15, replace=False), 'sleep_hours'] = np.nan   # inject missing values
df.loc[rng.choice(n, 3,  replace=False), 'allowance']   = 999      # inject a few outliers
print(df.shape); df.head()

**Reading the code, line by line:**
- we generate four numeric columns and one text column (`barangay`), with `quiz_score` genuinely
  driven by `study_hours` and `sleep_hours` plus noise;
- then we *deliberately* punch 15 missing sleep values and 3 absurd allowances (`999`) so the lab
  has gaps and outliers to catch;
- `df.head()` shows the first five rows &mdash; always your first look at a new dataset.

---
## Part 1 &middot; The first look: shape, types, summary

Three commands you run on *every* new dataset: `.info()` (columns + types + how many non-null),
and `.describe()` (min/max/mean/quartiles for the numbers).

In [ ]:
df.info()                    # (1) column names, data types, non-null counts
print()
df.describe().round(1)       # (2) numeric summary: count, mean, std, min, quartiles, max

**Reading the output:**
- **(1)** `.info()` shows `sleep_hours` has fewer non-null entries than the rest &mdash; that's the
  missing data. `barangay` is an `object` (text) column; the others are numbers.
- **(2)** `.describe()` &mdash; look at `allowance`: its **max is 999** while the mean is ~120. That
  gap between a sensible mean and a wild max is the fingerprint of an outlier.

**Answer here:**

1. From `.describe()`, which column has a max that looks impossible for a daily student allowance?
   &rarr; *your answer*

2. `.info()` shows `sleep_hours` with fewer non-nulls. Roughly how many values are missing?
   &rarr; *your answer*

---
## Part 2 &middot; Distributions: what does each column look like?

A **histogram** shows the shape of a numeric column; **`value_counts`** counts a categorical one.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].hist(df['quiz_score'], bins=20)                     # (1) numeric -> histogram
ax[0].set_title('quiz_score distribution'); ax[0].set_xlabel('score')
df['barangay'].value_counts().plot.bar(ax=ax[1], rot=0)   # (2) categorical -> bar of counts
ax[1].set_title('students per barangay')
plt.tight_layout(); plt.show()

**Reading the code, line by line:**
- **(1)** `plt.hist` buckets `quiz_score` into 20 bars &mdash; you can see its centre and spread at a
  glance (roughly bell-shaped here).
- **(2)** `value_counts()` tallies each barangay; `.plot.bar()` draws it. Categorical columns get
  counts, not histograms.

**Answer here:**

1. Is `quiz_score` roughly symmetric, or skewed to one side? One sentence.
   &rarr; *your answer*

---
## Part 3 &middot; Missing data: find it, then fill it

You can't feed `NaN` (missing) into most models. Step one is *finding* the gaps; step two is
deciding what to do. A common, safe choice is filling with the column's **median** (robust to
outliers, unlike the mean).

In [ ]:
print('missing per column:')
print(df.isna().sum())                                    # (1) count NaNs in each column

median_sleep = df['sleep_hours'].median()                 # (2) the fill value
df['sleep_hours'] = df['sleep_hours'].fillna(median_sleep) # (3) fill the gaps
print(f'\nfilled sleep_hours gaps with the median = {median_sleep:.2f}')
print('missing now:', df['sleep_hours'].isna().sum())

**Reading the code, line by line:**
- **(1)** `df.isna()` marks every cell True/False for missing; `.sum()` counts the Trues per column.
- **(2)** `.median()` ignores the NaNs and returns the middle sleep value.
- **(3)** `.fillna(...)` replaces every gap with that median. The recount is now 0.

**Answer here:**

1. Why fill with the **median** rather than the **mean** here (remember the `999` outliers)?
   &rarr; *your answer*

2. Filling is not free &mdash; you invented data. Name one situation where *dropping* the rows with
   missing values would be safer than filling.
   &rarr; *your answer*

---
## Part 4 &middot; Outliers: the IQR rule

An **outlier** is a point far from the rest. The classic rule uses the **interquartile range**
(IQR = Q3 &minus; Q1): flag anything below $Q_1 - 1.5\,\text{IQR}$ or above $Q_3 + 1.5\,\text{IQR}$.
A **boxplot** draws exactly this.

In [ ]:
q1, q3 = df['allowance'].quantile([0.25, 0.75])           # (1) the quartiles
iqr = q3 - q1
low, high = q1 - 1.5*iqr, q3 + 1.5*iqr                     # (2) the fences
outliers = df[(df['allowance'] < low) | (df['allowance'] > high)]
print(f'IQR fences: [{low:.0f}, {high:.0f}]  ->  {len(outliers)} outliers found')

plt.figure(figsize=(6,4)); plt.boxplot(df['allowance'], vert=False)
plt.title('allowance: the dots past the whisker are outliers'); plt.tight_layout(); plt.show()

**Reading the code, line by line:**
- **(1)** `.quantile([.25,.75])` gives Q1 and Q3; their difference is the IQR.
- **(2)** the fences are 1.5&times;IQR beyond each quartile; points outside are flagged. The boxplot
  shows the box (Q1&ndash;Q3), the median line, whiskers to the fences, and the `999`s as dots beyond.
- We *found* 3 outliers &mdash; exactly the ones we planted.

**Answer here:**

1. Would you delete these 3 rows, cap them at the fence, or investigate them? There's no single
   right answer &mdash; give your choice and one reason.
   &rarr; *your answer*

---
## Part 5 &middot; Correlation: which columns move together?

**Correlation** measures how strongly two numeric columns rise/fall together, from &minus;1 to +1.
`df.corr()` gives every pair at once; a **heatmap** makes the pattern pop. (We drop the outliers
first so they don't distort it.)

In [ ]:
clean = df[df['allowance'] < high]                        # (1) drop the 999 outliers
corr = clean[['study_hours','sleep_hours','allowance','quiz_score']].corr()   # (2) all pairs
print(corr.round(2))

plt.figure(figsize=(5.5,4.5))
im = plt.imshow(corr, vmin=-1, vmax=1, cmap='coolwarm')   # (3) colour = correlation
plt.colorbar(im)
plt.xticks(range(4), corr.columns, rotation=45, ha='right'); plt.yticks(range(4), corr.columns)
for i in range(4):                                         # (4) write the numbers on the cells
    for j in range(4):
        plt.text(j, i, f'{corr.iloc[i,j]:.2f}', ha='center', va='center')
plt.title('correlation heatmap'); plt.tight_layout(); plt.show()

**Reading the code, line by line:**
- **(1)** drop the outlier rows so one silly `999` doesn't warp the correlations.
- **(2)** `.corr()` returns a 4&times;4 table: each cell is the correlation of a pair (diagonal is 1,
  a column with itself).
- **(3)** `imshow` colours the table &mdash; red = strong positive, blue = negative, pale = near zero.
- **(4)** we print the number in each cell. Look at the `quiz_score` row: it's most red with
  `study_hours` &mdash; exactly the relationship we baked in.

**Answer here:**

1. Which single column is most strongly correlated with `quiz_score`? Does that match how we built
   the data?
   &rarr; *your answer*

2. `allowance` has near-zero correlation with `quiz_score`. In one sentence: what does a
   correlation near 0 tell you (and what does it *not* rule out)?
   &rarr; *your answer*

---
## Where you actually are

Set the pace honestly. Replace each `-` with: **solid** / **rusty** / **never really got it**.

| | You |
|---|---|
| pandas .head/.info/.describe | - |
| Histograms vs value_counts | - |
| Finding & filling missing values | - |
| The IQR outlier rule + boxplots | - |
| Reading a correlation heatmap | - |

**Which part took longest, and where did you get stuck?**
&rarr; *your answer*

**In one plain sentence: why look at the data before building any model?**
&rarr; *your answer*

---
## Stretch &mdash; optional

Required part is done; nothing below is graded.

### Stretch &middot; Group and compare

Does average quiz score differ by barangay? `groupby` answers it in one line. Fill it in.

In [ ]:
# your code here: print clean.groupby('barangay')['quiz_score'].mean().round(1)
# then: are the barangay averages close, or far apart? what would that suggest?


---
## Submitting

Run the cell below. It uploads this notebook straight from Colab &mdash; nothing to download.

You need a **submit token**: open
[https://portal.latarak.com/student/submit-token](https://portal.latarak.com/student/submit-token),
sign in, press the button, then paste it when the cell asks. The cell hides what you type.

In [ ]:
# --- Submit this notebook ------------------------------------------------------
# Colab only. Anywhere else, use the manual route described below this cell.
import getpass, json, urllib.request, urllib.error

PORTAL, COURSE, WEEK = "https://portal.latarak.com", "cmsc173", 5

try:
    from google.colab import _message
except ImportError:
    raise SystemExit(
        "Not running in Colab. Download this notebook "
        "(File > Download > Download .ipynb) and upload it at "
        "https://portal.latarak.com/course/cmsc173/lab/5/submit"
    )

nb = _message.blocking_request("get_ipynb", timeout_sec=90)["ipynb"]
token = getpass.getpass("Submit token (hidden as you type): ").strip()

req = urllib.request.Request(
    PORTAL + "/api/labs/" + COURSE + "/submit-notebook",
    data=json.dumps({"week": WEEK, "notebook": nb}).encode(),
    headers={"Content-Type": "application/json", "Authorization": "Bearer " + token},
    method="POST",
)
try:
    with urllib.request.urlopen(req, timeout=120) as r:
        out = json.load(r)
    print("Submitted", out["course"], "week", out["week"], "for", out["student"])
    print(out["cells"], "cells,", out["executed"], "executed")
    print(out["message"])
except urllib.error.HTTPError as e:
    print("Not submitted:", json.loads(e.read()).get("error", e.reason))

Prefer to do it by hand? **File &rarr; Download &rarr; Download .ipynb**, then go to the
[Week 5 submission page](https://portal.latarak.com/course/cmsc173/lab/5/submit) and upload it.

Blank cells are fine and guesses are fine. Don't polish this until it hides what you knew.